In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# 1. データの準備
# ---------------------------------------------------------
print("データを読み込んでいます...")

# 投球データ (CSV)
df_stats = pd.read_csv('daily_stats_cleaned.csv')
df_stats['game_date'] = pd.to_datetime(df_stats['game_date'])

# 怪我データ (SQLite)
conn = sqlite3.connect('baseball_analysis.db')
df_injuries = pd.read_sql('SELECT player_name, injury_date, body_part, side FROM injuries', conn)
conn.close()

# ---【修正ポイント】日付変換のエラー対策 ---
# "12月下旬" などの変換できない日付は、強制的に NaT (無効値) に変換する
df_injuries['injury_date'] = pd.to_datetime(df_injuries['injury_date'], errors='coerce')

# 日付が無効になってしまったデータを確認して除外
invalid_count = df_injuries['injury_date'].isna().sum()
if invalid_count > 0:
    print(f"⚠️ 注意: 日付形式が不明な {invalid_count} 件のデータを分析から除外しました。")
    # 除外実行
    df_injuries = df_injuries.dropna(subset=['injury_date'])

print(f"有効な怪我データ数: {len(df_injuries)} 件")

# ---------------------------------------------------------
# 2. 「怪我直前データ」の抽出ロジック
# ---------------------------------------------------------
print("怪我直前の登板データを抽出中...")

analysis_data = df_stats.copy()
analysis_data['condition'] = 'Normal' # デフォルトは「通常」

count_injury_games = 0

for _, injury in df_injuries.iterrows():
    name = injury['player_name']
    i_date = injury['injury_date']
    
    # この選手の全登板データを抽出
    player_games = analysis_data[analysis_data['pitcher_name_clean'] == name]
    
    # 「怪我した日から遡って30日以内」かつ「怪我日より前」の登板を探す
    target_games = player_games[
        (player_games['game_date'] < i_date) & 
        (player_games['game_date'] >= i_date - pd.Timedelta(days=30))
    ]
    
    if len(target_games) > 0:
        # 見つかった登板の状態を「Pre-Injury(怪我直前)」に書き換える
        analysis_data.loc[target_games.index, 'condition'] = 'Pre-Injury'
        count_injury_games += len(target_games)

print(f"分析対象: 怪我直前の登板数 {count_injury_games} 試合 vs 通常登板数 {len(analysis_data) - count_injury_games} 試合")

# もし怪我直前のデータが1件もなければ、グラフが出せないので終了
if count_injury_games == 0:
    print("❌ 怪我直前の登板データが見つかりませんでした。")
    print("理由: 怪我の日付より前の登板がCSVに含まれていないか、名前のマッチングがうまくいっていない可能性があります。")
    exit()

# ---------------------------------------------------------
# 3. グラフで比較・可視化
# ---------------------------------------------------------
# フォント設定（文字化けする場合は適宜変更してください）
# plt.rcParams['font.family'] = 'MS Gothic' 
sns.set_style("whitegrid")

metrics = {
    'pitch_count': 'Pitch Count (球数)',
    'rest_days': 'Rest Days (中日数)',
    'max_speed': 'Max Speed (最速)',
    'breaking_ball_ratio': 'Breaking Ball % (変化球割合)'
}

plt.figure(figsize=(15, 10))

for i, (col, title) in enumerate(metrics.items(), 1):
    plt.subplot(2, 2, i)
    
    # 箱ひげ図
    sns.boxplot(data=analysis_data, x='condition', y=col, showfliers=False, palette="Set2")
    # 平均値マーカー
    sns.pointplot(data=analysis_data, x='condition', y=col, errorbar=None, color='red', markers="D", join=False, scale=0.8)
    
    plt.title(f"Comparison of {title}")
    plt.xlabel("")
    plt.ylabel(col)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 4. 数値で確認（平均値の比較）
# ---------------------------------------------------------
summary = analysis_data.groupby('condition')[list(metrics.keys())].mean()
print("\n【平均値の比較】")
print(summary)

# 差分計算
try:
    diff = summary.loc['Pre-Injury'] - summary.loc['Normal']
    print("\n【怪我直前はどう変わった？ (差分)】")
    for col in metrics:
        val = diff[col]
        print(f"{col}: {val:+.2f}")
except KeyError:
    print("\n※データ不足のため差分を計算できませんでした。")